# Step 3: Create Genie Space & Save Session

Creates a Databricks Genie Space from the tables created in Step 2,
saves session metadata to a UC table, and writes state.json for the app.

In [ ]:
dbutils.widgets.text("catalog", "yd_launchpad_final_classic_catalog")
dbutils.widgets.text("schema", "genie_app")
dbutils.widgets.text("company_name", "NovaTech Logistics")
dbutils.widgets.text("primary_color", "#1a73e8")
dbutils.widgets.text("secondary_color", "#ea4335")
dbutils.widgets.text("warehouse_id", "551addcb4415adb7")

import re
def slugify(name: str) -> str:
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', name.lower()).strip('_')
    if slug and slug[0].isdigit():
        slug = f'co_{slug}'
    return slug[:50]

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
COMPANY_NAME = dbutils.widgets.get("company_name")
COMPANY_SLUG = slugify(COMPANY_NAME)
PRIMARY_COLOR = dbutils.widgets.get("primary_color")
SECONDARY_COLOR = dbutils.widgets.get("secondary_color")
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/{COMPANY_SLUG}"

print(f"Company: {COMPANY_NAME}")
print(f"Schema: {CATALOG}.{SCHEMA}")

In [ ]:
import json
import uuid
import requests

# Load pipeline state from previous steps
state_raw = dbutils.fs.head(f"{VOLUME_PATH}/pipeline_state.json", 100000)
state = json.loads(state_raw)

COMPANY_DESC = state["company_description"]
COMPANY_SLUG = state["company_slug"]
tables_info = state["tables_info"]
sample_questions = state.get("sample_questions", [])

TABLE_IDENTIFIERS = [t["full_name"] for t in tables_info]
DISPLAY_NAME = f"{COMPANY_NAME} Analytics"

# Get workspace host and token
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

print(f"Tables: {TABLE_IDENTIFIERS}")
print(f"Display name: {DISPLAY_NAME}")
print(f"Sample questions: {len(sample_questions)}")

In [ ]:
# Create Genie Space (v2 serialized_space format)
print(f"Creating Genie Space: {DISPLAY_NAME}")

# Get current user for parent_path
user_resp = requests.get(f"{host}/api/2.0/preview/scim/v2/Me", headers=headers)
user_resp.raise_for_status()
username = user_resp.json().get("userName", "")
parent_path = f"/Workspace/Users/{username}"

# Tables MUST be sorted alphabetically
sorted_tables = sorted(TABLE_IDENTIFIERS)

serialized = {
    "version": 2,
    "data_sources": {
        "tables": [{"identifier": tid} for tid in sorted_tables]
    },
    "config": {
        "sample_questions": [
            {"id": uuid.uuid4().hex, "question": [q]} for q in sample_questions
        ]
    },
    "instructions": {
        "text_instructions": [
            {"id": uuid.uuid4().hex, "content": [COMPANY_DESC]}
        ]
    },
}

payload = {
    "title": DISPLAY_NAME,
    "description": COMPANY_DESC,
    "warehouse_id": WAREHOUSE_ID,
    "parent_path": parent_path,
    "serialized_space": json.dumps(serialized),
}

resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload)
print(f"Status: {resp.status_code}")
print(f"Response: {resp.text}")
resp.raise_for_status()
space = resp.json()
space_id = space.get("space_id") or space.get("id", "")
print(f"\nCreated Genie Space: {space_id}")

# Grant app service principal access to the Genie Space
APP_SP = "677d1641-521c-4df6-91f4-dacea8be74e7"
perm_resp = requests.patch(
    f"{host}/api/2.0/permissions/genie/{space_id}",
    headers=headers,
    json={
        "access_control_list": [{
            "service_principal_name": APP_SP,
            "permission_level": "CAN_MANAGE"
        }]
    },
)
if perm_resp.ok:
    print(f"Granted CAN_MANAGE to app SP on Genie Space {space_id}")
else:
    print(f"WARNING: Could not grant permissions: {perm_resp.text}")

In [ ]:
# Create sessions metadata table if it doesn't exist
sessions_table = f"`{CATALOG}`.`{SCHEMA}`.`sessions`"
spark.sql(f"""CREATE TABLE IF NOT EXISTS {sessions_table} (
    session_id STRING NOT NULL COMMENT 'Unique session identifier',
    space_id STRING NOT NULL COMMENT 'Genie Space ID',
    company_name STRING NOT NULL COMMENT 'Company display name',
    description STRING COMMENT 'Company description',
    company_slug STRING NOT NULL COMMENT 'Company slug used as table prefix',
    logo_path STRING COMMENT 'Path to company logo',
    primary_color STRING COMMENT 'Brand primary color',
    secondary_color STRING COMMENT 'Brand secondary color',
    tables_json STRING COMMENT 'JSON array of table metadata',
    sample_questions_json STRING COMMENT 'JSON array of sample questions',
    created_at TIMESTAMP COMMENT 'When this session was created'
) COMMENT 'Metadata for all created Genie Spaces'""")

# Insert session record
session_id = uuid.uuid4().hex
tables_json = json.dumps(tables_info).replace("'", "''")
questions_json = json.dumps(sample_questions).replace("'", "''")
desc_escaped = COMPANY_DESC.replace("'", "''")
name_escaped = COMPANY_NAME.replace("'", "''")

spark.sql(f"""INSERT INTO {sessions_table}
    (session_id, space_id, company_name, description, company_slug,
     logo_path, primary_color, secondary_color, tables_json, sample_questions_json, created_at)
VALUES
    ('{session_id}', '{space_id}', '{name_escaped}', '{desc_escaped}', '{COMPANY_SLUG}',
     '', '{PRIMARY_COLOR}', '{SECONDARY_COLOR}', '{tables_json}', '{questions_json}', CURRENT_TIMESTAMP())""")

print(f"Session saved: {session_id}")
print(f"Sessions table: {sessions_table}")

In [ ]:
# Write state.json to volume for the app to read
app_state = {
    "space_id": space_id,
    "display_name": DISPLAY_NAME,
    "catalog": CATALOG,
    "schema_name": SCHEMA,
    "warehouse_id": WAREHOUSE_ID,
    "tables": tables_info,
    "sample_questions": sample_questions,
    "branding": {
        "company_name": COMPANY_NAME,
        "description": COMPANY_DESC,
        "logo_path": "",
        "primary_color": PRIMARY_COLOR,
        "secondary_color": SECONDARY_COLOR,
    },
}

state_json = json.dumps(app_state, indent=2)
dbutils.fs.put(f"{VOLUME_PATH}/state.json", state_json, overwrite=True)

print(f"\nState written to {VOLUME_PATH}/state.json")
print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE!")
print(f"  Space ID:   {space_id}")
print(f"  Session ID: {session_id}")
print(f"  Schema:     {CATALOG}.{SCHEMA}")
print(f"  Tables:     {[t['table_name'] for t in tables_info]}")
print(f"{'='*60}")